# Predição de resultados do Brasileirão com modelos de classificação

## 1. Importações e configurações

Define o dataset, a coluna alvo, a estratégia de separação treino/teste, a inclusão de empates e o classificador escolhido.

In [129]:
from pathlib import Path

import pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC

DATASET_PATH = Path("brasileirao_serie_a_2018_2023_v3.csv")
TARGET_COLUMN = "vencedor"
SPLIT_COLUMN = "ano_campeonato"

INCLUDE_DRAWS = True
DRAW_LABEL = "empate"

CLASSIFIER_NAME = "naive_bayes"
CLASSIFIERS = {
    "naive_bayes": GaussianNB(),
    "svm": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True),
    "hist_gradient_boosting": HistGradientBoostingClassifier(),
}

## 2. Construção do modelo

Infere colunas numéricas/categóricas e cria o pipeline com o classificador selecionado.

In [130]:
def infer_feature_columns(df: pd.DataFrame) -> list[str]:
    if TARGET_COLUMN not in df.columns:
        raise ValueError(f"Coluna alvo não encontrada no CSV: {TARGET_COLUMN}")

    return [column for column in df.columns if column != TARGET_COLUMN]


def infer_column_types(
    df: pd.DataFrame,
    feature_columns: list[str],
) -> tuple[list[str], list[str]]:
    numeric_features = df[feature_columns].select_dtypes(include="number").columns.tolist()
    categorical_features = [
        column for column in feature_columns if column not in numeric_features
    ]

    return numeric_features, categorical_features


def build_model(
    numeric_features: list[str],
    categorical_features: list[str],
) -> Pipeline:
    if CLASSIFIER_NAME not in CLASSIFIERS:
        available_classifiers = ", ".join(CLASSIFIERS)
        raise ValueError(
            f"Classificador inválido: {CLASSIFIER_NAME}. "
            f"Opções disponíveis: {available_classifiers}"
        )

    transformers = []

    if numeric_features:
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
            ]
        )
        transformers.append(("num", numeric_transformer, numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]
        )
        transformers.append(("cat", categorical_transformer, categorical_features))

    if not transformers:
        raise ValueError("Nenhuma coluna de feature foi encontrada para treinar o modelo.")

    preprocessor = ColumnTransformer(
        transformers=transformers,
        sparse_threshold=0.0,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", clone(CLASSIFIERS[CLASSIFIER_NAME])),
        ]
    )

## 3. Carregamento do dataset

Lê o CSV e valida se a coluna alvo configurada está presente.

In [131]:
def load_dataset() -> pd.DataFrame:
    if not DATASET_PATH.exists():
        raise FileNotFoundError(f"Dataset não encontrado: {DATASET_PATH}")

    df = pd.read_csv(DATASET_PATH)

    if TARGET_COLUMN not in df.columns:
        raise ValueError(f"Coluna alvo não encontrada no CSV: {TARGET_COLUMN}")

    return df

## 4. Preparação dos dados

Inclui ou remove empates conforme a configuração e separa treino/teste por coluna temporal quando disponível.

In [132]:
def prepare_data(
    df: pd.DataFrame,
    feature_columns: list[str],
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series, int, str]:
    working_df = df.copy()
    removed_draws_count = 0

    if not INCLUDE_DRAWS:
        removed_draws_count = int((working_df[TARGET_COLUMN] == DRAW_LABEL).sum())
        working_df = working_df[working_df[TARGET_COLUMN] != DRAW_LABEL].copy()

    if working_df[TARGET_COLUMN].nunique() < 2:
        raise ValueError("O dataset precisa ter pelo menos duas classes no target.")

    if SPLIT_COLUMN in working_df.columns:
        split_values = working_df[SPLIT_COLUMN]
        last_season = split_values.max()

        if is_numeric_dtype(split_values):
            train_df = working_df[split_values < last_season]
            test_df = working_df[split_values == last_season]
            split_description = f"{SPLIT_COLUMN} < {last_season} para treino; {SPLIT_COLUMN} == {last_season} para teste"
        else:
            train_df = working_df[split_values != last_season]
            test_df = working_df[split_values == last_season]
            split_description = f"{SPLIT_COLUMN} != {last_season} para treino; {SPLIT_COLUMN} == {last_season} para teste"
    else:
        raise ValueError(f"Coluna de split não encontrada no dataset: {SPLIT_COLUMN}")

    if train_df.empty:
        raise ValueError("Nenhum registro encontrado para treino.")

    if test_df.empty:
        raise ValueError("Nenhum registro encontrado para teste.")

    X_train = train_df[feature_columns]
    y_train = train_df[TARGET_COLUMN]
    X_test = test_df[feature_columns]
    y_test = test_df[TARGET_COLUMN]

    return X_train, y_train, X_test, y_test, removed_draws_count, split_description

## 5. Exemplos de predição

Monta uma tabela com registros do teste, resultado real, previsão e probabilidades por classe quando o classificador disponibiliza esse cálculo.

In [133]:
def show_prediction_examples(
    model: Pipeline,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    predictions: list[str],
    examples_count: int = 10,
) -> pd.DataFrame:
    results = X_test.copy()
    results["vencedor_real"] = y_test.values
    results["vencedor_previsto"] = predictions

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_test)
        classes = list(model.classes_)

        for index, class_name in enumerate(classes):
            results[f"prob_{class_name}"] = probabilities[:, index]

    return results.head(examples_count)

## 6. Treinamento e avaliação

Executa o fluxo principal: carrega os dados, infere features, treina o classificador selecionado e calcula métricas.

In [134]:
df = load_dataset()
feature_columns = infer_feature_columns(df)
numeric_features, categorical_features = infer_column_types(df, feature_columns)
X_train, y_train, X_test, y_test, removed_draws_count, split_description = prepare_data(
    df,
    feature_columns,
)

model = build_model(numeric_features, categorical_features)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(f"Modelo: {CLASSIFIER_NAME}")
print(f"Dataset: {DATASET_PATH.name}")
print(f"Target: {TARGET_COLUMN}")
print(f"Split: {split_description}")
print(f"Empates incluídos: {INCLUDE_DRAWS}")
print(f"Empates removidos: {removed_draws_count}")
print(f"Features numéricas: {len(numeric_features)}")
print(f"Features categóricas: {len(categorical_features)}")
print(f"Exemplos de treino: {len(X_train)}")
print(f"Exemplos de teste: {len(X_test)}")
print(f"Classes: {list(model.classes_)}")

accuracy = accuracy_score(y_test, predictions)
print(f"\nAcurácia no teste: {accuracy:.4f}")

Modelo: naive_bayes
Dataset: brasileirao_serie_a_2018_2023_v3.csv
Target: vencedor
Split: ano_campeonato < 2023 para treino; ano_campeonato == 2023 para teste
Empates incluídos: True
Empates removidos: 0
Features numéricas: 28
Features categóricas: 8
Exemplos de treino: 1758
Exemplos de teste: 288
Classes: [np.str_('casa'), np.str_('empate'), np.str_('fora')]

Acurácia no teste: 0.4792


## 7. Relatório de classificação

In [135]:
print(classification_report(y_test, predictions, digits=4))

              precision    recall  f1-score   support

        casa     0.5286    0.7929    0.6343       140
      empate     0.2778    0.0667    0.1075        75
        fora     0.3667    0.3014    0.3308        73

    accuracy                         0.4792       288
   macro avg     0.3910    0.3870    0.3575       288
weighted avg     0.4222    0.4792    0.4202       288



## 8. Matriz de confusão

In [136]:
confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=model.classes_),
    index=model.classes_,
    columns=model.classes_,
)

confusion_matrix_df

,casa,empate,fora
casa,111,10,19
empate,51,5,19
fora,48,3,22


## 9. Exemplos de predições

In [137]:
prediction_examples = show_prediction_examples(model, X_test, y_test, predictions)
prediction_examples

,ano_campeonato,mes_campeonato,data,rodada,time_mandante,time_visitante,estadio,PPJ_pre_jogo_mandante,PPJ_pre_jogo_visitante,xG_pre_jogo_mandante,...,colocacao_visitante,valor_equipe_titular_mandante,valor_equipe_titular_visitante,idade_media_titular_mandante,idade_media_titular_visitante,vencedor_real,vencedor_previsto,prob_casa,prob_empate,prob_fora
429,2023,6,05/06/2023,9,Vasco da Gama,Flamengo,Estádio São Januário,0.33,1.00,1.88,...,5,37600000,84700000,26.2,26.1,fora,fora,0.003048,0.135081,0.861872
702,2023,8,20/08/2023,20,Santos,Grêmio,Estádio Urbano Caldeira,1.40,1.25,1.39,...,5,37400000,40000000,27.2,28.5,casa,casa,0.396959,0.316379,0.286663
1467,2023,5,14/05/2023,6,Grêmio,Fortaleza,Arena do Grêmio,2.00,2.00,1.43,...,8,26900000,13350000,28.4,28.5,empate,casa,0.555451,0.276000,0.168549
1549,2023,4,15/04/2023,1,Palmeiras,Cuiabá,Allianz Parque,0.00,0.00,0.00,...,15,84200000,5930000,26.8,29.6,casa,casa,0.966970,0.020319,0.012711
1550,2023,4,15/04/2023,1,Fortaleza,Internacional,Estádio Comendador Agostinho Prada,0.00,0.00,0.00,...,10,12800000,32800000,30.7,27.6,empate,casa,0.360939,0.342998,0.296063
1551,2023,4,22/04/2023,2,Fluminense,Athletico-PR,Estádio Nilton Santos,0.00,0.00,0.00,...,11,36950000,27300000,30.7,27.9,casa,casa,0.551558,0.265469,0.182973
1552,2023,4,23/04/2023,2,Cruzeiro,Grêmio,Estádio Raimundo Sampaio,0.00,0.00,0.00,...,13,17400000,25100000,29.4,29.1,casa,casa,0.449908,0.318326,0.231766
1554,2023,4,16/04/2023,1,Flamengo,Coritiba,Estádio Nilton Santos,0.00,0.00,0.00,...,19,79650000,18000000,26.5,28.3,casa,casa,0.949320,0.030784,0.019897
1557,2023,4,15/04/2023,1,Botafogo,São Paulo,Estádio Nilton Santos,0.00,0.00,0.00,...,14,20800000,28400000,29.9,29.1,casa,casa,0.434238,0.322844,0.242918
1559,2023,4,23/04/2023,2,Vasco da Gama,Palmeiras,Estádio Nivaldo Pereira,0.00,0.00,0.00,...,4,49700000,67200000,25.5,26.6,empate,fora,0.073335,0.270151,0.656513
